In [1]:
# Task 1: In day13.ipynb, define the required fields for each file
from pathlib import Path

DATA_DIR = Path("../archive")
REQUIRED_FIELDS = {
    "olist_order_items_dataset.csv": [
        "order_id", "order_item_id", "product_id", "seller_id",
        "shipping_limit_date", "price", "freight_value",
    ],
    "olist_orders_dataset.csv": [
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_estimated_delivery_date",
    ],
    "olist_customers_dataset.csv": [
        "customer_id", "customer_unique_id", "customer_zip_code_prefix",
        "customer_city", "customer_state",
    ],
}
for name, fields in REQUIRED_FIELDS.items():
    print(f"{name}: {len(fields)} required fields -> {fields}")

olist_order_items_dataset.csv: 7 required fields -> ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
olist_orders_dataset.csv: 5 required fields -> ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_estimated_delivery_date']
olist_customers_dataset.csv: 5 required fields -> ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [2]:
# Task 2: Validate every row and collect errors
import csv


def read_rows(path, delimiter=","):
    """Return (fieldnames, rows) after reading a CSV file."""
    with open(path, newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle, delimiter=delimiter)
        if reader.fieldnames is None:
            raise ValueError(f"No header found in {path}")
        return reader.fieldnames, [dict(row) for row in reader]


def validate_row(row, required, line_number):
    """Collect the missing-field errors for a single row."""
    errors = []
    for field in required:
        value = row.get(field)
        if value is None or str(value).strip() == "":
            errors.append(f"line {line_number}: missing required field '{field}'")
    return errors


def validate_every_row(rows, required):
    """Validate every row and collect (line_number, row, errors) tuples."""
    results = []
    for line_number, row in enumerate(rows, start=2):  # line 1 is the header
        errors = validate_row(row, required, line_number)
        errors += check_types(row)
        errors += check_business_rules(row)
        errors += check_regex(row)
        results.append((line_number, row, errors))
    return results


print("read_rows / validate_row / validate_every_row defined")

read_rows / validate_row / validate_every_row defined


In [3]:
# Task 3: Write rejected rows to bad_records.csv with a reason
import csv


def write_bad_records(entries, path="bad_records.csv"):
    """Write rejected rows and the reason they were rejected to a CSV file."""
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["source_file", "source_line", "reason", "row_data"])
        for source_file, line_number, row, reason in entries:
            writer.writerow([source_file, line_number, reason, str(row)])
    return len(entries)


print("write_bad_records defined; output file: bad_records.csv")

write_bad_records defined; output file: bad_records.csv


In [4]:
# Task 4: Check data types (numeric price, parseable dates)
from datetime import datetime

DATE_FIELDS = (
    "shipping_limit_date", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date", "review_creation_date", "review_answer_timestamp",
)


def check_types(row):
    """Report non-numeric price/freight values and unparseable dates."""
    errors = []
    for field in ("price", "freight_value"):
        value = row.get(field)
        if value not in (None, ""):
            try:
                float(value)
            except (TypeError, ValueError):
                errors.append(f"'{field}' is not numeric: {value!r}")
    for field in DATE_FIELDS:
        value = row.get(field)
        if value not in (None, ""):
            try:
                datetime.strptime(value, "%Y-%m-%d %H:%M:%S")
            except (TypeError, ValueError):
                errors.append(f"'{field}' is not a parseable date: {value!r}")
    return errors


print("check_types defined")

check_types defined


In [5]:
# Task 5: Business rules: price > 0, order_status in the allowed list
ALLOWED_ORDER_STATUSES = {
    "delivered", "shipped", "canceled", "unavailable", "invoiced",
    "processing", "created", "approved",
}


def check_business_rules(row):
    """Apply the business rules for price and order status."""
    errors = []
    price = row.get("price")
    if price not in (None, ""):
        try:
            if float(price) <= 0:
                errors.append(f"price must be > 0, got {price!r}")
        except (TypeError, ValueError):
            pass  # the numeric type error is reported by check_types
    status = row.get("order_status")
    if status not in (None, "") and status not in ALLOWED_ORDER_STATUSES:
        errors.append(f"order_status {status!r} is not allowed")
    return errors


print("check_business_rules defined; allowed statuses:", sorted(ALLOWED_ORDER_STATUSES))

check_business_rules defined; allowed statuses: ['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']


In [6]:
# Task 6: Regex: zip prefix must be 5 digits; IDs must be 32 hex characters
import re

ZIP_RE = re.compile(r"^\d{5}$")
HEX_ID_RE = re.compile(r"^[0-9a-fA-F]{32}$")
ID_FIELDS = (
    "order_id", "customer_id", "customer_unique_id", "product_id",
    "seller_id", "review_id",
)
ZIP_FIELDS = (
    "customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix",
)


def check_regex(row):
    """Check zip prefixes and identifier formats with regular expressions."""
    errors = []
    for field in ZIP_FIELDS:
        value = row.get(field)
        if value not in (None, "") and not ZIP_RE.fullmatch(str(value)):
            errors.append(f"'{field}' must be a 5-digit zip prefix, got {value!r}")
    for field in ID_FIELDS:
        value = row.get(field)
        if value not in (None, "") and not HEX_ID_RE.fullmatch(str(value)):
            errors.append(f"'{field}' must be a 32-character hex id, got {value!r}")
    return errors


print("Patterns:", ZIP_RE.pattern, "|", HEX_ID_RE.pattern)

Patterns: ^\d{5}$ | ^[0-9a-fA-F]{32}$


In [7]:
# Task 7: Create custom exceptions: SchemaError and FileFormatError
class SchemaError(Exception):
    """Raised when a file is missing required columns or breaks the schema."""


class FileFormatError(Exception):
    """Raised when a file cannot be read or is not a usable CSV."""


def validate_schema(fieldnames, required):
    """Raise SchemaError when required columns are absent."""
    missing = [field for field in required if field not in fieldnames]
    if missing:
        raise SchemaError(f"Missing columns: {missing}")
    return True


print("SchemaError and FileFormatError defined")

SchemaError and FileFormatError defined


In [8]:
# Task 8: Handle a missing or empty file gracefully
def safe_read_csv(path, delimiter=","):
    """Read a file, returning empty results with a message instead of crashing."""
    try:
        fieldnames, rows = read_rows(path, delimiter)
    except FileNotFoundError:
        print(f"[SKIP] File not found: {path}")
        return [], []
    except (OSError, ValueError) as exc:
        raise FileFormatError(f"Could not read {path}: {exc}") from exc
    if not fieldnames or not rows:
        print(f"[SKIP] File is empty: {path}")
        return [], []
    return fieldnames, rows


try:
    safe_read_csv(DATA_DIR / "does_not_exist.csv")
except FileFormatError as exc:
    print(f"[SKIP] {exc}")

[SKIP] File not found: ..\archive\does_not_exist.csv


In [9]:
# Task 9: Print a summary: total, valid and rejected rows
SUMMARY = {}
bad_entries = []

for name, required in REQUIRED_FIELDS.items():
    path = DATA_DIR / name
    try:
        fieldnames, rows = safe_read_csv(path)
        if not rows:
            continue
        validate_schema(fieldnames, required)
    except (SchemaError, FileFormatError) as exc:
        print(f"[SKIP] {name}: {exc}")
        continue

    results = validate_every_row(rows, required)
    for line_number, row, errors in results:
        if errors:
            bad_entries.append((name, line_number, row, "; ".join(errors)))

    valid = sum(1 for _, _, errors in results if not errors)
    rejected = len(results) - valid
    SUMMARY[name] = {"total": len(rows), "valid": valid, "rejected": rejected}
    print(f"{name}: total={len(rows)}, valid={valid}, rejected={rejected}")

write_bad_records(bad_entries, "bad_records.csv")
print(f"Wrote {len(bad_entries)} rejected rows to bad_records.csv")

olist_order_items_dataset.csv: total=112650, valid=112650, rejected=0


olist_orders_dataset.csv: total=99441, valid=99441, rejected=0


olist_customers_dataset.csv: total=99441, valid=99441, rejected=0
Wrote 0 rejected rows to bad_records.csv


In [10]:
# Task 10: Fail the whole file if more than 20% of rows are rejected (configurable)
import json
from pathlib import Path

CONFIG_PATH = Path("validation_config.json")
DEFAULT_CONFIG = {"max_rejected_ratio": 0.20}
if CONFIG_PATH.exists():
    config = {**DEFAULT_CONFIG, **json.loads(CONFIG_PATH.read_text(encoding="utf-8"))}
else:
    config = DEFAULT_CONFIG
    CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")

threshold = config["max_rejected_ratio"]
print(f"Configured rejection threshold: {threshold:.0%}")

for name, stats in SUMMARY.items():
    ratio = stats["rejected"] / stats["total"] if stats["total"] else 0.0
    if ratio > threshold:
        raise SchemaError(
            f"{name}: {stats['rejected']}/{stats['total']} rows rejected "
            f"({ratio:.1%}) exceeds the {threshold:.0%} limit"
        )
    print(f"{name}: {ratio:.1%} rejected - within limit")

Configured rejection threshold: 20%
olist_order_items_dataset.csv: 0.0% rejected - within limit
olist_orders_dataset.csv: 0.0% rejected - within limit
olist_customers_dataset.csv: 0.0% rejected - within limit
